# Zero-shot vs few-shot

**Session 2 · small model (`llama3.2:3b`) vs big model (`gpt-oss:120b-cloud`)**

The folklore says few-shot always wins. Measure it. On a modern small model doing a familiar
task (support-ticket routing), adding examples often does **not** clear the noise — `compare()`
says INCONCLUSIVE. So: start zero-shot, and add examples only when the number tells you to.
Few-shot earns its keep on unfamiliar label schemes, strict output formats, and specific edge
cases — the "your turn" section builds one.

In [1]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
from utils import ask, SMALL_MODEL, BIG_MODEL
from eval import load_cases, compare, sweep_models, exact


### Worked example

Route support tickets to `billing` / `bug` / `other` over the 40-case set, with a tolerant
scorer (the label appears anywhere in the reply). Zero-shot vs a 5-example few-shot prompt,
`repeats=3`.

In [4]:
cases = load_cases("../eval/datasets/support_tickets.jsonl")
print(cases)

[{'input': 'I was charged twice this month', 'expected': 'billing'}, {'input': 'The app crashes when I tap export', 'expected': 'bug'}, {'input': 'Do you offer a student discount?', 'expected': 'other'}, {'input': 'Payment went through but my plan still says free', 'expected': 'billing'}, {'input': 'Export runs but the PDF comes out blank', 'expected': 'bug'}, {'input': 'How do I change my email address?', 'expected': 'other'}, {'input': 'My invoice shows the wrong VAT rate', 'expected': 'billing'}, {'input': 'The dark mode toggle does nothing', 'expected': 'bug'}, {'input': 'Is there an API rate limit?', 'expected': 'other'}, {'input': 'You charged me again after I cancelled', 'expected': 'billing'}, {'input': 'Search returns results from projects I deleted', 'expected': 'bug'}, {'input': 'Can I export my data to CSV?', 'expected': 'other'}, {'input': 'The report total does not match the sum of the line items', 'expected': 'bug'}, {'input': 'I need a copy of my receipt for June', 'exp

In [6]:
LABELS = ["billing", "bug", "other"]

def clean(out):
    out = out.strip().lower()
    return next((lab for lab in LABELS if lab in out), out)

ZERO = ('Classify the support ticket as billing, bug, or other.\n'
        'Reply with ONE lowercase word.\n\nTicket: "{t}" ->')

FEW = ('Classify the support ticket as billing, bug, or other.\n'
       'Reply with ONE lowercase word.\n\n'
       'Ticket: "I cannot log in since the update" -> bug\n'
       'Ticket: "Refund me for the double charge" -> billing\n'
       'Ticket: "What are your office hours?" -> other\n'
       'Ticket: "How do I update my payment card?" -> other\n'
       'Ticket: "The totals on the dashboard are wrong" -> bug\n\n'
       'Ticket: "{t}" ->')

def make(template, model):
    return lambda t: clean(ask(template.format(t=t), model=model))

print("small model:")
compare(cases, make(ZERO, SMALL_MODEL), make(FEW, SMALL_MODEL),
        labels=("zero-shot", "few-shot"), scorer=exact, repeats=3)


small model:
  zero-shot                  60%   (spread 0% over 3 runs)
  few-shot                   68%   (spread 0% over 3 runs)
  gap +8%   vs   run-to-run noise 0%   ->   REAL


{'a': {'n': 40,
  'repeats': 3,
  'accuracies': [0.6, 0.6, 0.6],
  'acc_mean': 0.6,
  'acc_min': 0.6,
  'acc_max': 0.6,
  'spread': 0.0,
  'accuracy': 0.6,
  'passed': 24,
  'results': [{'input': 'I was charged twice this month',
    'expected': 'billing',
    'output': 'bug',
    'pass': False},
   {'input': 'The app crashes when I tap export',
    'expected': 'bug',
    'output': 'bug',
    'pass': True},
   {'input': 'Do you offer a student discount?',
    'expected': 'other',
    'output': 'other',
    'pass': True},
   {'input': 'Payment went through but my plan still says free',
    'expected': 'billing',
    'output': 'bug',
    'pass': False},
   {'input': 'Export runs but the PDF comes out blank',
    'expected': 'bug',
    'output': 'bug',
    'pass': True},
   {'input': 'How do I change my email address?',
    'expected': 'other',
    'output': 'other',
    'pass': True},
   {'input': 'My invoice shows the wrong VAT rate',
    'expected': 'billing',
    'output': 'bug',
    

### Same examples, bigger model

`sweep_models` runs each prompt on both models. The big model's zero-shot already scores near
its few-shot number, so the examples buy little there — spend that prompt budget elsewhere.

In [ ]:
for shot, tmpl in [("zero-shot", ZERO), ("few-shot", FEW)]:
    print(f"\n{shot}:")
    sweep_models(cases, lambda m, t=tmpl: make(t, m), [SMALL_MODEL, BIG_MODEL],
                 scorer=exact, repeats=2)


## Your turn - vary the example

1. **Make few-shot win.** Invent a label scheme the model can't guess: priority `P1`/`P2`/`P3`
   with a specific rule ("P1 = paying customer blocked"). Zero-shot will flail; a few examples
   should pin the rule. Re-run `compare()` — now is it REAL?
2. Score on the **raw** output with `scorer=exact` and no `clean()`. Does few-shot help or hurt
   parseability here? (It can go either way with a `->` continuation prompt — measure.)
3. Record the small-model and big-model zero-vs-few gaps, with verdicts, in your commit message.